In [ ]:
# ======================================================================
# FEATURE CORRELATION ANALYSIS (14 Variables vs IMD Rain Anomaly)
# ======================================================================
import time
import numpy as np
import pandas as pd
import xarray as xr

t0 = time.time()
print("Loading datasets...")

# 1. Load original & new ECMWF datasets
ds_ecmv = xr.open_zarr("../data/raw/s2s_reforecast.zarr")
ds_new = xr.open_zarr("/Users/abhimanyu/Downloads/IFS_reforecast_download-main/s2s_new_vars.zarr")
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")["rain"].where(lambda x: x != -999)

# 2. Align new variables with original ECMWF spatial grid and time ordering
ds_new_aligned = ds_new.reindex(time=ds_ecmv.time).interp(lat=ds_ecmv.lat, lon=ds_ecmv.lon)
ds_combined = xr.merge([ds_ecmv, ds_new_aligned])

print(f"Combined dataset loaded in {time.time()-t0:.1f}s. Variables ({len(ds_combined.data_vars)}):")
for v in ds_combined.data_vars:
    print(" -", v)

# Prep IMD coordinates & target
imd = ds_imd.assign_coords(time=ds_imd["time"].dt.floor("D"))
flat_lat, flat_lon = imd["lat"].values, imd["lon"].values
H, W = len(flat_lat), len(flat_lon)

init_times = ds_combined["time"].values
leads = ds_combined["step"].values.astype(int)
WINDOWS = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]

# Select sample subset of years for fast analysis (e.g., 2015-2019)
sample_years_mask = np.isin(init_times.astype("datetime64[Y]").astype(int) + 1970, range(2015, 2020))
sub_combined = ds_combined.sel(time=sample_years_mask)
sub_init = init_times[sample_years_mask]

# 3. Helper functions: DOY Climatology Matrix & Pearson Correlation
def _clim_mat(values, doys, window=7):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    centers = np.arange(1, 367)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    M = (np.minimum(d, 366 - d) <= window).astype(np.float32)
    counts = M @ np.isfinite(v2).astype(np.float32)
    sums = M @ np.nan_to_num(v2).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        c = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return c.reshape((366,) + shp).astype(np.float32)

def corr_map(p, t, fin):
    """Per-cell Pearson Correlation r over time axis."""
    with np.errstate(invalid="ignore", divide="ignore"):
        pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
        tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
        num = np.nansum(np.where(fin, (p - pm) * (t - tm), np.nan), axis=0)
        den = np.sqrt(np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0)
                      * np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0))
        return np.where(den > 0, num / den, np.nan)

# 4. Compute correlation per window and per feature
results = []
for wname, lo, hi in WINDOWS:
    sel_lead = np.where((leads >= lo) & (leads <= hi))[0]
    
    # Window mean of predictor variables
    X_w = sub_combined.isel(step=sel_lead).mean(dim="step").compute()
    
    # Calculate valid target times (init + step)
    step_td = sub_combined["step"].isel(step=sel_lead).values.astype("timedelta64[D]")
    vt = sub_init[:, None] + step_td[None, :]
    
    # Interpolate all predictor features to IMD high-res grid
    X_fine = X_w.interp(lat=flat_lat, lon=flat_lon)
    
    # IMD Target rain over lead window
    y_all = imd.reindex(time=vt.ravel()).astype(np.float32).compute().values
    ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
    
    centre_date = sub_init + np.timedelta64((lo + hi) // 2, "D")
    doya = xr.DataArray(centre_date, dims="t").dt.dayofyear.values
    
    # Target anomalies
    clim_y = _clim_mat(ya, doya)
    y_anom = ya - clim_y[doya - 1]
    
    mask = np.isfinite(y_anom).all(axis=0)
    fin = np.isfinite(y_anom) & mask[None]
    
    # Compute correlation map for each predictor variable
    for var in ds_combined.data_vars:
        x_val = X_fine[var].values.astype(np.float32)
        clim_x = _clim_mat(x_val, doya)
        x_anom = x_val - clim_x[doya - 1]
        
        r_map = corr_map(x_anom, y_anom, fin)
        mean_r = float(np.nanmean(r_map[mask]))
        abs_mean_r = float(np.nanmean(np.abs(r_map[mask])))
        
        results.append({
            "Window": wname,
            "Variable": var,
            "Mean_Corr": mean_r,
            "Mean_Abs_Corr": abs_mean_r
        })

# 5. Display Summary Rankings
df_res = pd.DataFrame(results)
print("\n=================================================================================")
print("FEATURE CORRELATION RANKING WITH IMD RAINFALL ANOMALY")
print("=================================================================================")

for wname in ["week2", "week3_4", "week5_6"]:
    print(f"\n>>> WINDOW: {wname.upper()} <<<")
    sub = df_res[df_res["Window"] == wname].sort_values(by="Mean_Abs_Corr", ascending=False)
    print(f"{'Rank':<5} | {'Variable Name':<32} | {'Mean r':<10} | {'Mean |r|':<10}")
    print("-" * 65)
    for rank, (_, row) in enumerate(sub.iterrows(), 1):
        is_new = "*" if ("geopotential" in row["Variable"] or "thermal" in row["Variable"]) else " "
        print(f"{rank:<5} | {row['Variable'] + is_new:<32} | {row['Mean_Corr']:<+10.4f} | {row['Mean_Abs_Corr']:<10.4f}")

print("\n* = newly added feature")


In [ ]:
# ======================================================================
# BOXPLOT VISUALIZATION: SPATIAL CORRELATION OF ALL 26 FEATURE VARIABLES
# ======================================================================
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

t0 = time.time()
print("Loading datasets...")

# 1. Open original dataset (21 features) and new dataset (5 features)
ds_old = xr.open_zarr("../data/processed/s2s_reforecast_sorted.zarr")
ds_new = xr.open_zarr("/Users/abhimanyu/Downloads/IFS_reforecast_download-main/s2s_new_vars.zarr")
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")["rain"].where(lambda x: x != -999)

# Align new variables to original grid
ds_new_aligned = ds_new.reindex(time=ds_old.time).interp(lat=ds_old.lat, lon=ds_old.lon)
ds_combined = xr.merge([ds_old, ds_new_aligned])

print(f"Total combined feature variables ({len(ds_combined.data_vars)}):")
for idx, v in enumerate(ds_combined.data_vars, 1):
    print(f" {idx:>2}. {v}")

# 2. Prep IMD coordinates & target
imd = ds_imd.assign_coords(time=ds_imd["time"].dt.floor("D"))
flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

init_times = ds_combined["time"].values
leads = ds_combined["step"].values.astype(int)
WINDOWS = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]

# Select sample subset of years (e.g., 2015-2019) for fast calculation
sample_years_mask = np.isin(init_times.astype("datetime64[Y]").astype(int) + 1970, range(2015, 2020))
sub_combined = ds_combined.sel(time=sample_years_mask)
sub_init = init_times[sample_years_mask]

# 3. Climatology Matrix & Pearson Correlation Functions
def _clim_mat(values, doys, window=7):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    centers = np.arange(1, 367)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    M = (np.minimum(d, 366 - d) <= window).astype(np.float32)
    counts = M @ np.isfinite(v2).astype(np.float32)
    sums = M @ np.nan_to_num(v2).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        c = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return c.reshape((366,) + shp).astype(np.float32)

def corr_map(p, t, fin):
    with np.errstate(invalid="ignore", divide="ignore"):
        pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
        tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
        num = np.nansum(np.where(fin, (p - pm) * (t - tm), np.nan), axis=0)
        den = np.sqrt(np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0)
                      * np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0))
        return np.where(den > 0, num / den, np.nan)

# 4. Compute per-cell correlation across all 26 variables & 3 windows
cell_records = []

for wname, lo, hi in WINDOWS:
    print(f"Processing window: {wname}...")
    sel_lead = np.where((leads >= lo) & (leads <= hi))[0]
    X_w = sub_combined.isel(step=sel_lead).mean(dim="step").compute()
    
    step_td = sub_combined["step"].isel(step=sel_lead).values.astype("timedelta64[D]")
    vt = sub_init[:, None] + step_td[None, :]
    X_fine = X_w.interp(lat=flat_lat, lon=flat_lon)
    
    y_all = imd.reindex(time=vt.ravel()).astype(np.float32).compute().values
    ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
    
    centre_date = sub_init + np.timedelta64((lo + hi) // 2, "D")
    doya = xr.DataArray(centre_date, dims="t").dt.dayofyear.values
    
    clim_y = _clim_mat(ya, doya)
    y_anom = ya - clim_y[doya - 1]
    
    mask = np.isfinite(y_anom).all(axis=0)
    fin = np.isfinite(y_anom) & mask[None]
    
    for var in ds_combined.data_vars:
        x_val = X_fine[var].values.astype(np.float32)
        clim_x = _clim_mat(x_val, doya)
        x_anom = x_val - clim_x[doya - 1]
        
        r_map = corr_map(x_anom, y_anom, fin)
        valid_r = r_map[mask]
        
        for r_val in valid_r:
            cell_records.append({
                "Window": wname,
                "Variable": var,
                "Corr": r_val
            })

df_all = pd.DataFrame(cell_records)

# 5. Boxplot Visualization
plt.figure(figsize=(14, 12))

# Order variables by median absolute correlation in weeks 3-4
w34 = df_all[df_all["Window"] == "week3_4"]
order = w34.groupby("Variable")["Corr"].apply(lambda x: x.abs().median()).sort_values(ascending=False).index

sns.boxplot(
    data=df_all, 
    y="Variable", 
    x="Corr", 
    hue="Window", 
    order=order, 
    showfliers=False, 
    palette={"week2": "#1f77b4", "week3_4": "#ff7f0e", "week5_6": "#2ca02c"}
)

plt.axvline(0, color="black", linestyle="--", alpha=0.7, lw=1)
plt.title("Spatial Anomaly Correlation (r) across All 26 Feature Variables vs IMD Rain", fontsize=13, fontweight="bold")
plt.xlabel("Pearson Correlation Coefficient (r) across India Land Cells", fontsize=11)
plt.ylabel("Feature Variable", fontsize=11)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(title="Lead Window", loc="lower right")

plt.tight_layout()
plt.savefig("../results/figures/all_26_features_correlation_boxplot.png", dpi=150)
plt.show()


In [ ]:
# ======================================================================
# GENERATE PER-LEAD-DAY CORRELATION BOXPLOTS (Day 0 to Day 42)
# ======================================================================
import os
import time
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

output_dir = "../results/diagnostics/lead_day_corr_boxplots"
os.makedirs(output_dir, exist_ok=True)

print("Loading datasets...")
ds_old = xr.open_zarr("../data/raw/s2s_reforecast.zarr")
ds_new = xr.open_zarr("/Users/abhimanyu/Downloads/IFS_reforecast_download-main/s2s_new_vars.zarr")
ds_imd = xr.open_zarr("../data/raw/IMD_rainfall_0p25.zarr")["rain"].where(lambda x: x != -999)

# Align new variables
ds_new_aligned = ds_new.reindex(time=ds_old.time).interp(lat=ds_old.lat, lon=ds_old.lon)
ds_combined = xr.merge([ds_old, ds_new_aligned])

print(f"Total feature variables ({len(ds_combined.data_vars)})")

# Prep IMD grid & valid mask
imd = ds_imd.assign_coords(time=ds_imd["time"].dt.floor("D"))
flat_lat, flat_lon = imd["lat"].values, imd["lon"].values
H, W = len(flat_lat), len(flat_lon)

init_times = ds_combined["time"].values
steps = ds_combined["step"].values.astype(int)

# Use sample subset of years (e.g., 2015-2019) for fast processing
sample_years_mask = np.isin(init_times.astype("datetime64[Y]").astype(int) + 1970, range(2015, 2020))
sub_combined = ds_combined.sel(time=sample_years_mask)
sub_init = init_times[sample_years_mask]

# Helper: Climatology Matrix
def _clim_mat(values, doys, window=7):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    centers = np.arange(1, 367)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    M = (np.minimum(d, 366 - d) <= window).astype(np.float32)
    counts = M @ np.isfinite(v2).astype(np.float32)
    sums = M @ np.nan_to_num(v2).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        c = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return c.reshape((366,) + shp).astype(np.float32)

# Helper: Pearson Correlation Map over time
def corr_map(p, t, fin):
    with np.errstate(invalid="ignore", divide="ignore"):
        pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
        tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
        num = np.nansum(np.where(fin, (p - pm) * (t - tm), np.nan), axis=0)
        den = np.sqrt(np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0)
                      * np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0))
        return np.where(den > 0, num / den, np.nan)

print(f"Generating boxplots for all {len(steps)} lead days in '{output_dir}/'...")

# Loop through each lead day
for step in steps:
    t_step_start = time.time()
    
    # Extract predictors at this exact lead step
    X_step = sub_combined.sel(step=step).compute()
    
    # Target valid times (init_time + step days)
    step_td = np.timedelta64(step, "D")
    vt = sub_init + step_td
    
    # Interpolate predictor features to fine IMD grid
    X_fine = X_step.interp(lat=flat_lat, lon=flat_lon)
    
    # Get target IMD rain at valid dates
    ya = imd.reindex(time=vt).astype(np.float32).compute().values
    
    doya = xr.DataArray(vt, dims="t").dt.dayofyear.values
    clim_y = _clim_mat(ya, doya)
    y_anom = ya - clim_y[doya - 1]
    
    mask = np.isfinite(y_anom).all(axis=0)
    fin = np.isfinite(y_anom) & mask[None]
    
    records = []
    for var in ds_combined.data_vars:
        x_val = X_fine[var].values.astype(np.float32)
        clim_x = _clim_mat(x_val, doya)
        x_anom = x_val - clim_x[doya - 1]
        
        r_map = corr_map(x_anom, y_anom, fin)
        valid_r = r_map[mask]
        
        for r_val in valid_r:
            records.append({"Variable": var, "Corr": r_val})
            
    df_step = pd.DataFrame(records)
    
    # Plot boxplot for this specific lead day
    plt.figure(figsize=(12, 10))
    order = df_step.groupby("Variable")["Corr"].apply(lambda x: x.abs().median()).sort_values(ascending=False).index
    
    sns.boxplot(data=df_step, y="Variable", x="Corr", order=order, showfliers=False, color="#1f77b4")
    plt.axvline(0, color="black", linestyle="--", alpha=0.7, lw=1)
    plt.title(f"Spatial Anomaly Correlation (r) Distribution — Lead Day {step}", fontsize=13, fontweight="bold")
    plt.xlabel("Pearson Correlation Coefficient (r) across India Land Cells", fontsize=11)
    plt.ylabel("Feature Variable", fontsize=11)
    plt.xlim(-0.6, 0.6)
    plt.grid(True, linestyle=":", alpha=0.6)
    
    plt.tight_layout()
    out_path = os.path.join(output_dir, f"lead_day_{step:02d}.png")
    plt.savefig(out_path, dpi=120)
    plt.close()
    print(f"Saved lead_day_{step:02d}.png ({time.time() - t_step_start:.1f}s)")

print(f"\nAll {len(steps)} lead day boxplots saved in '{output_dir}/'!")
